# Fake News Detection with DistilBERT (Google Colab)
**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.
Download the ISOT dataset from Kaggle ("Fake and Real News Dataset" by Clément Bisaillon). Unzip to get `Fake.csv` and `True.csv`.
Then run the cells below one by one, in order.

## Cell 1: Install libraries

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

## Cell 2: Import libraries

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DistilBertTokenizerFast, DistilBertForSequenceClassification,
                          DataCollatorWithPadding,
                          TrainingArguments, Trainer)

print("GPU available:", torch.cuda.is_available())

## Cell 3: Upload and load the CSV files
A file picker opens. Select both `Fake.csv` and `True.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()

fake_df = pd.read_csv("Fake.csv")
true_df = pd.read_csv("True.csv")
print("Fake rows:", len(fake_df), "| True rows:", len(true_df))

## Cell 4: Combine files and create labels
FAKE = 0, REAL = 1 (models need numbers, not words).

In [ ]:
fake_df["label"] = 0
true_df["label"] = 1

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)   # shuffle
print(df.shape)
df.head()

## Cell 5: Basic data exploration

In [ ]:
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nClass counts:\n", df["label"].value_counts())

sns.countplot(x="label", data=df)
plt.xticks([0, 1], ["FAKE", "REAL"])
plt.title("Class distribution")
plt.show()

## Cell 6: Clean the text
We remove the `CITY (Reuters) -` tag, because nearly all REAL articles have it and the model could cheat by learning only that. Duplicates are removed **before** splitting to avoid data leakage.

In [ ]:
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df["content"] = df["title"] + ". " + df["text"]

def clean_text(t):
    t = re.sub(r"^.{0,80}?\(Reuters\)\s*-?\s*", "", t)
    t = re.sub(r"\bReuters\b", "", t)
    t = re.sub(r"http\S+", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["content"] = df["content"].apply(clean_text)
df = df[df["content"].str.len() > 50]
df = df.drop_duplicates(subset="content").reset_index(drop=True)

# 'subject' and 'date' leak the label, so we drop them
df = df[["content", "label"]]
print("Rows after cleaning:", len(df))
print(df["label"].value_counts())

## Cell 7: Reduce dataset size (recommended for beginners)
Use 6000 if Colab is slow or runs out of memory. Set `None` to use everything.

In [ ]:
SAMPLE_SIZE = 10000

if SAMPLE_SIZE:
    df = pd.concat([
        df[df["label"] == 0].sample(SAMPLE_SIZE // 2, random_state=42),
        df[df["label"] == 1].sample(SAMPLE_SIZE // 2, random_state=42),
    ]).sample(frac=1, random_state=42).reset_index(drop=True)
print(df["label"].value_counts())

## Cell 8: Split into train / validation / test (70 / 15 / 15)
The test set is used **only once**, at the end.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])
print("Train:", len(train_df), "| Validation:", len(val_df), "| Test:", len(test_df))

## Cell 9: Select Model Architecture & Tokenize
Choose between `distilbert-base-uncased` (fast, 66M params) or `bert-base-uncased` (standard, 110M params).

In [ ]:
# Model architecture checkpoint:
MODEL_CHECKPOINT = "distilbert-base-uncased"  # or "bert-base-uncased"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(batch):
    return tokenizer(batch["content"], truncation=True, max_length=MAX_LEN)

def make_dataset(frame):
    ds = Dataset.from_pandas(frame.reset_index(drop=True))
    return ds.map(tokenize, batched=True)

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)


## Cell 10: Load pretrained Transformer (DistilBERT / BERT)
Loads the model with a classification head for 2 classes: 0 (FAKE) and 1 (REAL).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label={0: "FAKE", 1: "REAL"},
    label2id={"FAKE": 0, "REAL": 1},
)


## Cell 11: Fine-tune
Takes about 5-15 minutes on a T4 GPU. If you run out of memory: lower `SAMPLE_SIZE`, `MAX_LEN`, batch size to 8, epochs to 1, then restart the runtime.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds),
            "precision": precision, "recall": recall, "f1": f1}

args_dict = dict(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)
try:
    training_args = TrainingArguments(eval_strategy="epoch", **args_dict)
except TypeError:   # older transformers versions
    training_args = TrainingArguments(evaluation_strategy="epoch", **args_dict)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

## Cell 12: Final evaluation on the held-out test set
- **Accuracy**: of all articles, how many were right?
- **Precision**: when the model says REAL, how often is it correct?
- **Recall**: of all truly REAL articles, how many did it find?
- **F1-score**: one number balancing precision and recall.
- **Confusion matrix**: diagonal = correct, other cells = mistakes.

In [ ]:
pred_output = trainer.predict(test_ds)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}\n")
print(classification_report(y_true, y_pred, target_names=["FAKE", "REAL"]))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["FAKE", "REAL"], yticklabels=["FAKE", "REAL"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()

## Cell 13: Save the model and tokenizer

In [ ]:
import os
SAVE_DIR = "fake_news_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(os.listdir(SAVE_DIR))

## Cell 14 (Option A): Zip and download to your computer

In [ ]:
!zip -r fake_news_model.zip fake_news_model
from google.colab import files
files.download("fake_news_model.zip")

## Cell 15 (Option B, recommended): Upload to Hugging Face
1. Make a free account at huggingface.co.
2. Settings → Access Tokens → New token → type **Write** → copy it.
3. Change `your-username` below, run the cell, and paste the token when asked.

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login, HfApi

login()
REPO_ID = "your-username/fake-news-distilbert"   # <-- change this

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path="fake_news_model", repo_id=REPO_ID)
print("Done: https://huggingface.co/" + REPO_ID)